In [1]:
# ==========================================================
# Notebook 07: Access Control and Secure Search
# ==========================================================

import ast
import re

import faiss
import numpy as np
import pandas as pd

from rank_bm25 import BM25Okapi

from sentence_transformers import SentenceTransformer, CrossEncoder

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
DATA_PATH = "data/processed/enterprise_corpus_acl.csv"

enterprise_df = pd.read_csv(DATA_PATH)

enterprise_df.head()

,source,title,content,author,doc_id,department,created_date,tags,acl_roles,security_level,owner_team
0,Slack,Phoenix Deployment Discussion,The Phoenix-2026 deployment is scheduled for n...,Alice,DOC0001,Engineering,2026-01-11,"['enterprise-search', 'semantic-search']","['Engineering', 'Public']",Internal,Engineering
1,Slack,Model Training Update,The semantic search model has been retrained u...,Bob,DOC0002,Finance,2026-01-12,"['phoenix', 'kubernetes']","['Finance', 'Public']",Restricted,Finance
2,Confluence,Phoenix Architecture Wiki,Project Phoenix-2026 follows a microservice ar...,Charlie,DOC0003,HR,2026-01-13,"['semantic-search', 'enterprise-search']","['HR', 'Public']",Restricted,HR
3,Confluence,Enterprise Search Design,Hybrid search combines BM25 keyword retrieval ...,Diana,DOC0004,Engineering,2026-01-14,"['fastapi', 'acl']","['Engineering', 'Public']",Internal,Engineering
4,Jira,PHX-245 Database Migration,Complete PostgreSQL migration before Phoenix p...,Eric,DOC0005,Engineering,2026-01-15,"['enterprise-search', 'kubernetes']","['Engineering', 'Public']",Internal,Engineering


In [3]:
def parse_acl(value):

    if isinstance(value, str):

        return ast.literal_eval(value)

    return value


enterprise_df["acl_roles"] = enterprise_df["acl_roles"].apply(parse_acl)

In [4]:
user_profiles = {
    "alice": {"department": "Engineering", "roles": ["Engineering", "Public"]},
    "bob": {"department": "HR", "roles": ["HR", "Public"]},
    "carol": {"department": "Finance", "roles": ["Finance", "Public"]},
    "david": {"department": "Legal", "roles": ["Legal", "Public"]},
}

user_profiles

{'alice': {'department': 'Engineering', 'roles': ['Engineering', 'Public']},
 'bob': {'department': 'HR', 'roles': ['HR', 'Public']},
 'carol': {'department': 'Finance', 'roles': ['Finance', 'Public']},
 'david': {'department': 'Legal', 'roles': ['Legal', 'Public']}}

In [5]:
def acl_filter(dataframe, user_roles):

    mask = dataframe["acl_roles"].apply(
        lambda acl: any(role in acl for role in user_roles)
    )

    return dataframe[mask].reset_index(drop=True)

In [6]:
alice_docs = acl_filter(enterprise_df, user_profiles["alice"]["roles"])

alice_docs[["doc_id", "title", "acl_roles"]]

,doc_id,title,acl_roles
0,DOC0001,Phoenix Deployment Discussion,"[Engineering, Public]"
1,DOC0002,Model Training Update,"[Finance, Public]"
2,DOC0003,Phoenix Architecture Wiki,"[HR, Public]"
3,DOC0004,Enterprise Search Design,"[Engineering, Public]"
4,DOC0005,PHX-245 Database Migration,"[Engineering, Public]"
5,DOC0006,SEC-104 ACL Enhancement,"[HR, Public]"


In [7]:
secure_df = acl_filter(enterprise_df, user_profiles["alice"]["roles"])

secure_df["search_text"] = secure_df["title"] + " " + secure_df["content"]

In [8]:
def preprocess_text(text):

    text = text.lower()

    text = re.sub(r"[^a-zA-Z0-9\s\-]", " ", text)

    return text.split()

In [9]:
tokenized_secure_corpus = [preprocess_text(doc) for doc in secure_df["search_text"]]

secure_bm25 = BM25Okapi(tokenized_secure_corpus)

In [10]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

secure_embeddings = embedding_model.encode(
    secure_df["search_text"].tolist(), convert_to_numpy=True, show_progress_bar=True
)

secure_embeddings = np.array(secure_embeddings).astype("float32")

faiss.normalize_L2(secure_embeddings)

secure_faiss = faiss.IndexFlatIP(secure_embeddings.shape[1])

secure_faiss.add(secure_embeddings)

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
def secure_hybrid_search(query, user_name, top_k=5):

    user_roles = user_profiles[user_name]["roles"]

    authorized_docs = acl_filter(enterprise_df, user_roles)

    authorized_docs = authorized_docs.copy()

    authorized_docs["search_text"] = (
        authorized_docs["title"] + " " + authorized_docs["content"]
    )

    return authorized_docs[["doc_id", "title", "department", "acl_roles"]].head(top_k)

In [20]:
secure_results = secure_hybrid_search(
    query="Phoenix deployment", user_name="alice", top_k=5
)

secure_results

,doc_id,title,department,acl_roles
0,DOC0001,Phoenix Deployment Discussion,Engineering,"[Engineering, Public]"
1,DOC0002,Model Training Update,Finance,"[Finance, Public]"
2,DOC0003,Phoenix Architecture Wiki,HR,"[HR, Public]"
3,DOC0004,Enterprise Search Design,Engineering,"[Engineering, Public]"
4,DOC0005,PHX-245 Database Migration,Engineering,"[Engineering, Public]"


In [21]:
secure_results_hr = secure_hybrid_search(
    query="salary policy", user_name="bob", top_k=5
)

secure_results_hr

,doc_id,title,department,acl_roles
0,DOC0001,Phoenix Deployment Discussion,Engineering,"[Engineering, Public]"
1,DOC0002,Model Training Update,Finance,"[Finance, Public]"
2,DOC0003,Phoenix Architecture Wiki,HR,"[HR, Public]"
3,DOC0004,Enterprise Search Design,Engineering,"[Engineering, Public]"
4,DOC0005,PHX-245 Database Migration,Engineering,"[Engineering, Public]"


In [23]:
# Example tenant assignment
enterprise_df["tenant_id"] = [
    "company_a",
    "company_a",
    "company_b",
    "company_b",
    "company_a",
    "company_b",
]

In [24]:
user_tenant = "company_a"

In [25]:
filtered_docs = enterprise_df[enterprise_df["tenant_id"] == user_tenant]

filtered_docs[["doc_id", "title", "tenant_id"]]

,doc_id,title,tenant_id
0,DOC0001,Phoenix Deployment Discussion,company_a
1,DOC0002,Model Training Update,company_a
4,DOC0005,PHX-245 Database Migration,company_a


In [27]:
import pickle
import os

OUTPUT_PATH = "artifacts/"

os.makedirs(OUTPUT_PATH, exist_ok=True)

security_config = {
    "acl_enabled": True,
    "pre_filtering": True,
    "multi_tenant_ready": True,
}

with open(OUTPUT_PATH + "security_config.pkl", "wb") as file:

    pickle.dump(security_config, file)

print("Security configuration saved!")

Security configuration saved!
